# 🤖 Умный чат-бот (OpenAI) для Google Colab

**Перед запуском:** слева в Colab нажми 🔑 **Секреты** → «Добавить секрет» → имя `OPENAI_API_KEY`, значение — твой ключ `sk-proj-...` → включи переключатель «Доступ из блокнота».
Ключ нельзя вставлять прямо в код.

In [ ]:
!pip install -q openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"   # можно заменить на другую модель из твоего аккаунта
SYSTEM_PROMPT = (
    "Ты умный, дружелюбный ассистент. Отвечай на языке пользователя, "
    "понятно и по делу. Если не уверен — честно скажи об этом."
)
MAX_TURNS = 20          # сколько последних реплик помнит бот

history = []            # история диалога

In [ ]:
def ask(user_text: str) -> str:
    """Отправляет сообщение, печатает ответ по мере генерации и сохраняет историю."""
    history.append({"role": "user", "content": user_text})
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history[-MAX_TURNS * 2:]

    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        stream=True,
    )

    answer = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            piece = chunk.choices[0].delta.content
            print(piece, end="", flush=True)
            answer += piece
    print()

    history.append({"role": "assistant", "content": answer})
    return answer

In [ ]:
print("Чат запущен. Команды: /reset — очистить память, /exit — выйти.\n")

while True:
    user_text = input("Ты: ").strip()
    if not user_text:
        continue
    if user_text.lower() in ("/exit", "exit", "выход"):
        print("Пока! 👋")
        break
    if user_text.lower() == "/reset":
        history.clear()
        print("Память очищена.\n")
        continue
    print("Бот: ", end="")
    try:
        ask(user_text)
    except Exception as e:
        print(f"\n⚠️ Ошибка: {e}")
    print()